# FIFA 2026 World Cup Predictor: Model Training & Tournament Simulation

This notebook provides a complete walk-through of the data merging, feature engineering, model training, Platt calibration, historical backtesting (2018 and 2022), and Monte Carlo tournament simulations for the FIFA 2026 World Cup.

### Run Environment Check
If you are running this in Google Colab, execute the cell below to clone the repository, install the dependencies, and set up the directory structure.

In [ ]:
# Environment setup for Google Colab
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running on Google Colab. Cloning repository...")
    !git clone https://github.com/mohamedazimal27/fifa2026-predictor.git
    %cd fifa2026-predictor
    !pip install -r requirements.txt
else:
    print("Running in local environment. Ensure you have installed requirements via 'pip install -r requirements.txt'")

## Section 1: Data Loading & Inspection

Let's load the primary match history dataset (Jürisoo's results) and check the cached Elo ratings data.

In [ ]:
import pandas as pd
import numpy as np
import os

results_path = "data/results.csv"
shootouts_path = "data/shootouts.csv"

df_results = pd.read_csv(results_path)
df_shootouts = pd.read_csv(shootouts_path)

print(f"Results Dataset: {df_results.shape[0]} matches, {df_results.shape[1]} columns.")
print(f"Shootouts Dataset: {df_shootouts.shape[0]} matches, {df_shootouts.shape[1]} columns.")
print("\nSample results:")
display(df_results.tail(3))

## Section 2: Feature Engineering & Preprocessing

Now we merge Jürisoo's match outcomes with the historical daily Elo ratings from `eloratings.net` and build features (Elo differences, decay form, interim coach flags, and host advantages).

In [ ]:
from src.data_pipeline.data_loader import merge_elo_and_results
from src.features import build_features

# Check if merged data is already cached to speed up iteration
features_cache_path = "data/processed_features.csv"

if os.path.exists(features_cache_path):
    print("Loading pre-computed features from cache...")
    df_feat = pd.read_csv(features_cache_path)
    df_feat['date'] = pd.to_datetime(df_feat['date'])
else:
    print("Merging Jürisoo matches + Elo ratings...")
    df_merged = merge_elo_and_results(results_path, "data/canonical_teams.json", "data/elo")
    
    print("Building temporal features (decay form, coach/squad features, host advantage)...")
    df_feat = build_features(df_merged, curated_path="data/curated_teams.json", half_life_days=365.0)
    df_feat.to_csv(features_cache_path, index=False)

print(f"\nFeatures constructed: {df_feat.shape[0]} rows, {df_feat.shape[1]} columns.")
print("Available features for model training:")
feature_cols = [
    'elo_diff', 'home_elo', 'away_elo',
    'home_form', 'away_form', 'form_diff',
    'home_squad_quality', 'away_squad_quality', 'squad_quality_diff',
    'home_interim_coach', 'away_interim_coach',
    'home_advantage', 'host_advantage_home', 'host_advantage_away'
]
for col in feature_cols:
    print(f" - {col}")

## Section 3: Model Training & Platt Calibration

We split our data chronologically to ensure no future leakage:
- **Train Set**: 2000–2015
- **Val B Set**: 2016–2017 (for Platt calibration fitting)
- **Test Set**: 2018 (World Cup backtesting)
- **Holdout Set**: 2022 (World Cup holdout validation)

We train an XGBoost classifier with sample weights and then apply Platt Scaling (Multiclass Logistic Regression) to calibrate the output probabilities.

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
import pickle

# Create splits
train_mask = (df_feat['date'] >= '2000-01-01') & (df_feat['date'] <= '2015-12-31')
val_b_mask = (df_feat['date'] >= '2016-01-01') & (df_feat['date'] <= '2017-12-31')
test_mask = (df_feat['date'] >= '2018-01-01') & (df_feat['date'] <= '2018-12-31')
holdout_mask = (df_feat['date'] >= '2022-01-01') & (df_feat['date'] <= '2022-12-31')

X_train, y_train = df_feat[train_mask][feature_cols], df_feat[train_mask]['target']
X_val_b, y_val_b = df_feat[val_b_mask][feature_cols], df_feat[val_b_mask]['target']
X_test, y_test = df_feat[test_mask][feature_cols], df_feat[test_mask]['target']
X_holdout, y_holdout = df_feat[holdout_mask][feature_cols], df_feat[holdout_mask]['target']

# Calculate balanced sample weights for training set
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Train base model
base_model = XGBClassifier(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)
base_model.fit(X_train, y_train, sample_weight=sample_weights)

# Train Platt Calibrator on Val B
val_b_preds = base_model.predict_proba(X_val_b)
calibrator = LogisticRegression(solver='lbfgs', C=1.0, random_state=42)
calibrator.fit(val_b_preds, y_val_b)

def predict_calibrated(X):
    base_preds = base_model.predict_proba(X)
    return calibrator.predict_proba(base_preds)

# Evaluate log loss
print("--- Log Loss Comparison (Uncalibrated vs. Calibrated) ---")
print(f"  Train:     {log_loss(y_train, base_model.predict_proba(X_train)):.4f}  ->  {log_loss(y_train, predict_calibrated(X_train)):.4f}")
print(f"  Val B:     {log_loss(y_val_b, base_model.predict_proba(X_val_b)):.4f}  ->  {log_loss(y_val_b, predict_calibrated(X_val_b)):.4f}")
print(f"  Test:      {log_loss(y_test, base_model.predict_proba(X_test)):.4f}  ->  {log_loss(y_test, predict_calibrated(X_test)):.4f}")
print(f"  Holdout:   {log_loss(y_holdout, base_model.predict_proba(X_holdout)):.4f}  ->  {log_loss(y_holdout, predict_calibrated(X_holdout)):.4f}")

# Save model
os.makedirs("models", exist_ok=True)
with open("models/fifa_model.pkl", 'wb') as f:
    pickle.dump({
        "base_model": base_model,
        "calibrator": calibrator,
        "feature_cols": feature_cols,
        "half_life_days": 365.0
    }, f)
print("\nCalibrated model successfully saved to 'models/fifa_model.pkl'.")

## Section 4: Backtesting 2018 World Cup

Let's look at the predictions and metrics (Brier Score, Log Loss) specifically for matches played during the 2018 World Cup.

In [ ]:
from sklearn.metrics import brier_score_loss

# Filter 2018 World Cup matches specifically
df_wc_2018 = df_feat[test_mask & (df_feat['tournament'] == 'FIFA World Cup')]
if df_wc_2018.empty:
    df_wc_2018 = df_feat[test_mask] # fallback to all test matches if tournament string differs

X_wc_2018 = df_wc_2018[feature_cols]
y_wc_2018 = df_wc_2018['target']

preds_2018 = predict_calibrated(X_wc_2018)
loss_2018 = log_loss(y_wc_2018, preds_2018)

# Multiclass Brier score is sum of squared differences divided by number of samples
y_onehot = pd.get_dummies(y_wc_2018).reindex(columns=[0, 1, 2], fill_value=0).values
brier_2018 = np.mean(np.sum((preds_2018 - y_onehot) ** 2, axis=1))

print(f"2018 World Cup Backtest results:")
print(f"  Number of matches: {len(df_wc_2018)}")
print(f"  Log Loss:         {loss_2018:.4f}")
print(f"  Brier Score:      {brier_2018:.4f}")
print(f"  Accuracy:         {accuracy_score(y_wc_2018, np.argmax(preds_2018, axis=1)):.2%}")

## Section 5: Backtesting 2022 World Cup

Next, we backtest the calibrated model on the 2022 World Cup hold-out set to evaluate its predictive performance.

In [ ]:
df_wc_2022 = df_feat[holdout_mask & (df_feat['tournament'] == 'FIFA World Cup')]
if df_wc_2022.empty:
    df_wc_2022 = df_feat[holdout_mask]

X_wc_2022 = df_wc_2022[feature_cols]
y_wc_2022 = df_wc_2022['target']

preds_2022 = predict_calibrated(X_wc_2022)
loss_2022 = log_loss(y_wc_2022, preds_2022)

y_onehot_2022 = pd.get_dummies(y_wc_2022).reindex(columns=[0, 1, 2], fill_value=0).values
brier_2022 = np.mean(np.sum((preds_2022 - y_onehot_2022) ** 2, axis=1))

print(f"2022 World Cup Backtest results:")
print(f"  Number of matches: {len(df_wc_2022)}")
print(f"  Log Loss:         {loss_2022:.4f}")
print(f"  Brier Score:      {brier_2022:.4f}")
print(f"  Accuracy:         {accuracy_score(y_wc_2022, np.argmax(preds_2022, axis=1)):.2%}")

## Section 6: 2026 World Cup Tournament Simulation

Let's run a full Monte Carlo simulation of the 48-team 2026 World Cup bracket using our calibrated model, Elo scores, squad ratings, coach data, and fatigue rules.

In [ ]:
from src.simulator import TournamentSimulator
import time

print("Initializing tournament simulator...")
t0 = time.time()
sim = TournamentSimulator()
print(f"Simulator initialized in {time.time() - t0:.2f} seconds (Matchup cache warmed!).")

print("Running 10,000 Monte Carlo tournament simulations...")
t0 = time.time()
probs = sim.run_monte_carlo(num_simulations=10000)
print(f"Finished 10,000 simulations in {time.time() - t0:.2f} seconds.")

# Sort teams by champion probability
sorted_probs = sorted(probs.items(), key=lambda x: x[1]['champion'], reverse=True)
print("\nTop 10 predicted champions:")
for idx, (team, stages) in enumerate(sorted_probs[:10]):
    print(f"  {idx+1}. {team:<15} Champ: {stages['champion']:.2%} | Runner-up: {stages['runner_up']:.2%} | 3rd: {stages['third_place']:.2%} | KO Stage: {1 - stages['group_stage_exit']:.2%}")

## Section 7: Interactive Visualizations

Let's visualize the results using Plotly. We will create:
1. Top 15 teams by Championship probability.
2. A stacked progression probability chart for the top 10 teams.
3. Model feature importance chart from our trained XGBoost classifier.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Chart 1: Champion Probability
top_15_teams = [t for t, _ in sorted_probs[:15]]
top_15_champs = [probs[t]['champion'] * 100 for t in top_15_teams]

fig1 = px.bar(
    x=top_15_champs,
    y=top_15_teams,
    orientation='h',
    labels={'x': 'Championship Probability (%)', 'y': 'Team'},
    title='FIFA 2026 World Cup: Top 15 Championship Probabilities',
    color=top_15_champs,
    color_continuous_scale='Viridis'
)
fig1.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
fig1.show()

# Chart 2: Stacked Progression Probability for Top 10 Teams
top_10_teams = [t for t, _ in sorted_probs[:10]]
stages = ['group_stage_exit', 'r32_exit', 'r16_exit', 'qf_exit', 'sf_exit', 'third_place', 'runner_up', 'champion']
stage_labels = ['Group Stage Exit', 'R32 Exit', 'R16 Exit', 'QF Exit', 'SF Exit', 'Third Place', 'Runner-up', 'Champion']

fig2 = go.Figure()
for stage, label in zip(stages, stage_labels):
    fig2.add_trace(go.Bar(
        name=label,
        x=top_10_teams,
        y=[probs[team][stage] * 100 for team in top_10_teams]
    ))

fig2.update_layout(
    barmode='stack',
    title='World Cup 2026 Progression Probabilities by Team (Top 10)',
    xaxis_title='Team',
    yaxis_title='Probability (%)',
    legend_title='Exit Stage',
    height=600
)
fig2.show()

# Chart 3: Feature Importance
importances = base_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values(by='Importance', ascending=True)

fig3 = px.bar(
    feat_imp_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='XGBoost Feature Importances',
    labels={'Importance': 'Relative Importance', 'Feature': 'Feature Name'},
    color='Importance',
    color_continuous_scale='Plasma'
)
fig3.show()